In [1]:
import warnings
warnings.filterwarnings('ignore')

# Vector Store

## Pdf to embeddings

In [8]:
import PyPDF2

filename = "./seguros/AF-SL-TablasGarantiasHogar-Eficaz-Modalidad13-v5-CAS.pdf"

texts = []
with open(filename, 'rb') as file:
    reader = PyPDF2.PdfReader(file)
    num_pags = len(reader.pages)
    print(num_pags)
    for page in range(num_pags):
        page_obj = reader.pages[page]
        tt = page_obj.extract_text()
        texts.append(tt)

4


In [13]:
# texts[1]

In [14]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [16]:
tt = texts[1]
tt_embeddings = embed_model.encode(tt)

In [19]:
len(tt_embeddings)

384

In [20]:
embeddings = embed_model.encode(texts)

In [25]:
len(embeddings)

4

## Indexing

In [26]:
import faiss

d = 384
index = faiss.IndexFlatL2(d)
faiss.normalize_L2(embeddings)
index.add(embeddings)

In [27]:
query = 'cristal roto'
query_embeddings = embed_model.encode(query)

In [29]:
query_embeddings.shape

(384,)

In [31]:
import numpy as np

query_embeddings = np.array([query_embeddings])
query_embeddings.shape

(1, 384)

In [34]:
distancias, documentos = index.search(query_embeddings, k=4)
print(distancias[0])
print(documentos[0])

[18.565748 23.05386  23.356403 24.12189 ]
[3 0 1 2]


# OpenAI

In [37]:
from openai import OpenAI
client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-4.1-nano",
    input="Dime las coberturas de un seguro de hogar en relacion a espejos rotos"
)

print(response.output_text)

Las coberturas de un seguro de hogar en relación a espejos rotos pueden variar dependiendo de la póliza y la aseguradora, pero generalmente incluyen las siguientes:

1. **Cobertura por daños accidentales**: 
   - Cubre la reparación o reemplazo de espejos rotos que hayan sido dañados accidentalmente dentro del inmueble, como un espejo de pared en el baño, habitación o salón.

2. **Reparación de daños por accidentes**:
   - Incluye incidentes como golpes o caídas que hayan provocado la rotura del espejo, siempre que sean considerados accidentes cubiertos en la póliza.

3. **Cobertura de cristalería**:
   - Algunas pólizas incluyen coberturas para cristalería en general, lo que puede abarcar espejos, vitrinas, ventanas y otros elementos de vidrio.

4. **Indemnización por reemplazo**:
   - En caso de rotura del espejo, el seguro puede cubrir el costo del reemplazo, incluyendo la mano de obra si es necesario.

Es importante revisar los detalles específicos de la póliza, ya que algunas aseg

In [39]:
# response

In [40]:
user_query = "Dime las coberturas de un seguro de hogar en relacion a espejos rotos"

In [52]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    temperature=0,
    messages=[
        {"role": "system", "content": "Eres un asistente de seguros muy mal hablado. Tu objetivo es responder preguntas del cliente."},
        {"role": "user", "content": user_query}
    ]
)

In [53]:
print(completion.choices[0].message.content)

¡Claro, pero no te hagas ilusiones! La mayoría de los seguros de hogar cubren daños accidentales, y eso incluye espejos rotos, siempre y cuando no sea por negligencia o uso indebido. La cobertura suele ser bajo la sección de daños a la propiedad o contenido, y en algunos casos, puede requerir que tengas una póliza que incluya protección contra riesgos específicos. Pero ojo, muchas veces hay límites y deducibles, así que revisa bien la letra pequeña. ¿Quieres que te ayude a buscar una póliza que cubra específicamente espejos rotos?


In [58]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    temperature=1,
    messages=[
        {"role": "system", "content": "Eres un asistente de seguros muy mal hablado. Tu objetivo es responder preguntas del cliente."},
        {"role": "user", "content": user_query}
    ]
)

In [59]:
print(completion.choices[0].message.content)

¡Claro, amigo! En un seguro de hogar, las coberturas por espejos rotos suelen estar incluidas dentro de la póliza de contenido o del hogar. Esto quiere decir que si se rompe un espejo por accidente, algunas pólizas cubren la reparación o el reemplazo. Pero ojo, no todo es color de rosa: muchas veces tienen deducibles o límites en la cobertura. O sea, no esperes que te paguen la colección completa si tienes un espejo de oro de 10 mil euros. Te recomiendo que revises tu póliza para ver exactamente qué detalles cubren y qué no. Si quieres, te puedo ayudar a entender mejor las condiciones específicas de tu seguro.


In [60]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    temperature=1,
    messages=[
        {"role": "system", "content": "Eres un asistente de seguros muy mal hablado. Tu objetivo es responder preguntas del cliente."},
        {"role": "user", "content": user_query}
    ]
)

In [61]:
print(completion.choices[0].message.content)

Mira, en un seguro de hogar, las coberturas por espejos rotos suelen estar incluidas en la sección de daños por incidentes accidentales. Pero ojo, no todos los pólizas cubren eso, así que debes revisar tu contrato específico. Normalmente, si rompes un espejo accidentalmente, algunas pólizas sí te cubren la reparación o reemplazo, pero otras solo cubren daños a bienes estructurales. O sea, revisa bien los detalles, porque si fue por negligencia o causa externa, quizá no tenga esa cobertura.


# Prompting

## Ejemplo

In [62]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_home.csv")
df = df[(df.stars==1) | (df.stars==5)].sample(5, random_state=42)
df

,stars,review_body,review_title,product_category
24140,5,Ocupa poco espacio con una prestación estupenda.,Sencillo y funcional,home
518,1,"No sirven para nada no son impermeables, no cu...",Pésimo y un engaño en rosa regla,home
1090,1,"De las 5 que vienen en el pack, tenía 2 en uso...",No merecen la pena,home
4474,1,"No se puede hacer nudo, se desliza",Hilo,home
23816,5,Buen material y medidas prácticas.,VARIAS MEDIDAS,home


In [73]:
def classification_openai(user_query):
    
    system_prompt = """
        Eres un clasificador de reviews de productos.
        Vas a recibir un Texto en español y tendras que clasificarlo como POSITIVO o NEGATIVO.
        Cuando encuentres un sentimiento positivo en el Texto, lo clasificas como POSITIVO.
        Cuando encuentres un sentimiento negativo en el Texto, lo clasificas como NEGATIVO.
        Siempre responde en español.
        Como respuesta solo darás la palabra POSITIVO o bien NEGATIVO. Nunca ambos.
        Si no sabes como clasificarlo, no responderás nada. No escribas más de lo permitido.
        """
        
    completion = client.chat.completions.create(
        model="gpt-4.1-nano",
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]
    )
    return completion.choices[0].message.content

In [74]:
reviews = list(df.review_body)
stars = list(df.stars)

for review in reviews:
    user_query = f"¿podrías clasificar esta review como positiva o negativa? {review}"
    output = classification_openai(user_query)
    print(output)

POSITIVO
NEGATIVO
NEGATIVO
NEGATIVO
POSITIVO


# RAG (manual)

In [75]:
user_query = "tengo alguna garantia que cubra la rotura de cristales?"

In [77]:
# retrieval

query_embeddings = embed_model.encode(user_query)
query_embeddings = np.array([query_embeddings])
_, doc = index.search(query_embeddings, k=1)

In [78]:
# augementation + generation

def response_openai(user_query, doc):
    
    system_prompt = """
        Eres un agente de seguros.
        Tienes que dar una RESPUESTA a una PREGUNTA de un usuario en base a un DOCUMENTO de coberturas.
        Utiliza un tono educado para dar la respuesta.
        Si el DOCUMENTO no puede dar respuesta a la PREGUNTA, \
        responde de forma escueta "NO hay información suficiente en el documento".
        La RESPUESTA debe ser breve. Siempre en español. 
        Quita espacios al principio y al final de la respuesta.
        """
        
    prompt = f"""
    PREGUNTA: {user_query}\n
    --DOCUMENTO--
    {doc}
    -------------
    RESPUESTA:
    """
        
    completion = client.chat.completions.create(
        model="gpt-4.1-nano",
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return completion.choices[0].message.content

In [112]:
texto = texts[1]

In [113]:
response = response_openai(user_query, texto)

In [114]:
response

'Sí, en tu seguro se cubre la rotura de cristales.'

In [115]:
texto

'HOGAR EFICAZ\nGARANTÍAS BÁSICAS EJEMPLOS\nIncendio y riesgos complementarios\nDaños en la vivienda y lo que contenga producidos por el fuego, una \nexplosión, la caída de un rayo y la acción del humo que se provoque ya tenga \nsu origen en la propia vivienda o externo, así como la reconstrucción del \njardín hasta el 5 % del importe que tengas asegurado como continente.Todos sabemos que estamos cubiertos en caso de incendio \npero, ¿sabías también que lo estás si se produce una llamarada \nde una sartén, quema la campana y mancha el techo de humo? \nPagamos la reparación de la campana y la pintura del techo.\nHumo\nNos hacemos cargo de los daños que ocasione el humo por escapes \nrepentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos.Por ejemplo, la limpieza de la pared ennegrecida por una fuga \nde humo puntual del tiro de la chimenea.\nDaños por agua\nDaños por fugas de agua en tuberías o depósitos o por no cerrar llaves o \ngrifos. También se cubren los gastos d